# NPU matrix multiplication

Load the verified overlay, tile logical M/N dimensions over the physical array,
and check every result against NumPy. K must fit the loaded hardware MAX_K.


In [ ]:
from pathlib import Path
import json
import os
import sys
import time

import numpy as np

start = Path.cwd().resolve()
repository_root = next((p for p in (start, *start.parents)
                        if (p / 'src/runtime/__init__.py').is_file()), None)
if repository_root is None:
    raise RuntimeError('Open the notebook from the source checkout or extracted package')
example_root = repository_root
sys.path.insert(0, str(repository_root))
from src.runtime import load_pynq_runtime


In [ ]:
artifact_dir = Path(os.environ.get('NPU_MATRIX_ARTIFACTS', str(repository_root / 'artifacts')))
if not artifact_dir.is_dir():
    artifact_dir = repository_root / 'build' / 'vivado' / 'npu_matrix_16x16' / 'artifacts'
manifest_path = artifact_dir / 'npu_matrix.manifest.json'
from src.vendors.npu_pynq.driver.overlay.verify import verify_artifacts
manifest = verify_artifacts(artifact_dir)
runtime = load_pynq_runtime(artifact_dir / 'npu_matrix.bit')
print({
    'source_commit': manifest['source_commit'],
    'vivado_version': manifest['vivado_version'],
    'target_part': manifest['target_part'],
    'physical_limits': (runtime.max_m, runtime.max_n, runtime.max_k),
})

In [ ]:
def tiled_matmul(a_matrix, b_matrix, software_timeout=10.0):
    """A @ B on the NPU: tile M and N over the physical array; K must fit MAX_K."""
    m, k = a_matrix.shape
    n = b_matrix.shape[1]
    if k > runtime.max_k:
        raise ValueError(f'K={k} exceeds the physical MAX_K={runtime.max_k}')
    output = np.empty((m, n), dtype=np.int32)
    tile_count = 0
    start = time.monotonic()
    for row in range(0, m, runtime.max_m):
        for column in range(0, n, runtime.max_n):
            a_tile = np.ascontiguousarray(a_matrix[row:row + runtime.max_m, :], dtype=np.int8)
            b_tile = np.ascontiguousarray(b_matrix[:, column:column + runtime.max_n], dtype=np.int8)
            output[row:row + runtime.max_m, column:column + runtime.max_n] = runtime.run(
                a_tile, b_tile, software_timeout=software_timeout)
            tile_count += 1
    elapsed = time.monotonic() - start
    operation_count = 2 * m * n * k
    return output, {
        'shape': (m, k, n),
        'tile_count': tile_count,
        'elapsed_seconds': elapsed,
        'operation_count': operation_count,
        'operations_per_second': operation_count / elapsed if elapsed > 0 else float('inf'),
    }

def reference(a_matrix, b_matrix):
    return (a_matrix.astype(np.int64) @ b_matrix.astype(np.int64)).astype(np.int32)

def run_and_check(label, a_matrix, b_matrix):
    output, metrics = tiled_matmul(a_matrix, b_matrix)
    np.testing.assert_array_equal(output, reference(a_matrix, b_matrix))
    print({'case': label, **metrics})
    return output, metrics


In [ ]:
normal_a = ((np.arange(runtime.max_m * 5).reshape(runtime.max_m, 5) % 17) - 8).astype(np.int8)
normal_b = ((np.arange(5 * runtime.max_n).reshape(5, runtime.max_n) % 19) - 9).astype(np.int8)
_, normal = run_and_check('full physical array', normal_a, normal_b)
assert normal['tile_count'] == 1


## Cross-tile multiplication

Multiply (M+1) x 5 by 5 x (N+1), where M/N are the loaded physical array
limits. This produces four physical jobs and exercises edge tiles.


In [ ]:
non_aligned_a = ((np.arange((runtime.max_m + 1) * 5).reshape(runtime.max_m + 1, 5) % 17) - 8).astype(np.int8)
non_aligned_b = ((np.arange(5 * (runtime.max_n + 1)).reshape(5, runtime.max_n + 1) % 19) - 9).astype(np.int8)
non_aligned, metrics = tiled_matmul(non_aligned_a, non_aligned_b)
np.testing.assert_array_equal(non_aligned, reference(non_aligned_a, non_aligned_b))
assert non_aligned.shape == (runtime.max_m + 1, runtime.max_n + 1)
assert non_aligned.dtype == np.int32
assert metrics['tile_count'] == 4
print({
    'A shape': non_aligned_a.shape,
    'B shape': non_aligned_b.shape,
    'C shape': non_aligned.shape,
    'tile_count': metrics['tile_count'],
    'result': non_aligned.tolist(),
})


In [ ]:
run_and_check('repeated', -non_aligned_a, non_aligned_b)
print('PASS: NPU matrix multiplication example')
